# Atividade 6
Desocupação por sexo (2012 x 2026) e horas semanais de afazeres domésticos (2022).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt   # corrigido: antes era 'import matplotlib as plt'
import seaborn as sb

sb.set_theme(style="whitegrid")

## 1. Carregando os dados

In [ ]:
# CSVs: separador ';' e decimal ','
d2012 = pd.read_csv("Tabela5-sem_emprego_2012__1_.csv", sep=";", decimal=",", encoding="utf-8-sig")
d2026 = pd.read_csv("Tabela5-sem_emprego_2026__1_.csv", sep=";", decimal=",", encoding="utf-8-sig")

# nomes de colunas mais simples
d2012.columns = ["Sigla", "Codigo", "Estado", "Homens_2012", "Mulheres_2012"]
d2026.columns = ["Sigla", "Codigo", "Estado", "Homens_2026", "Mulheres_2026"]

desocupados = d2012.merge(d2026[["Sigla", "Homens_2026", "Mulheres_2026"]], on="Sigla")
desocupados["Var_Mulheres"] = desocupados["Mulheres_2026"] - desocupados["Mulheres_2012"]
desocupados.head()

In [ ]:
# Planilha do IBGE (precisa do pacote xlrd para .xls: pip install xlrd)
horas = pd.read_excel("Tabela_1_1_1__1_.xls", header=None, skiprows=8, nrows=33)
horas.columns = ["Local", "Total", "Total_Branca", "Total_PretaParda",
                 "Homem_Branca", "Homem_PretaParda", "Mulher_Branca", "Mulher_PretaParda"]

# mantém só estados (remove Brasil e Grandes Regiões)
nao_estados = ["Brasil", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
horas = horas[~horas["Local"].isin(nao_estados)].copy()
horas["Homem_Media"] = horas[["Homem_Branca", "Homem_PretaParda"]].mean(axis=1)
horas["Mulher_Media"] = horas[["Mulher_Branca", "Mulher_PretaParda"]].mean(axis=1)
horas.head()

## 2. Percentual de mulheres entre os desocupados: 2012 x 2026

In [ ]:
ordem = desocupados.sort_values("Mulheres_2026", ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(ordem))
ax.bar([i - 0.2 for i in x], ordem["Mulheres_2012"], width=0.4, label="2012 T1")
ax.bar([i + 0.2 for i in x], ordem["Mulheres_2026"], width=0.4, label="2026 T1")
ax.axhline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xticks(list(x))
ax.set_xticklabels(ordem["Sigla"])
ax.set_ylabel("% de mulheres entre desocupados")
ax.set_title("Mulheres entre os desocupados por estado")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Horas semanais de afazeres domésticos: homens x mulheres (2022)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sb.scatterplot(data=horas, x="Homem_Media", y="Mulher_Media", ax=ax)
for _, r in horas.iterrows():
    ax.annotate(r["Local"], (r["Homem_Media"], r["Mulher_Media"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Horas semanais - homens")
ax.set_ylabel("Horas semanais - mulheres")
ax.set_title("Afazeres domésticos por sexo (2022)")
plt.tight_layout()
plt.show()

In [ ]:
horas["Diferenca"] = horas["Mulher_Media"] - horas["Homem_Media"]
print("Média nacional da diferença (mulheres - homens):", round(horas["Diferenca"].mean(), 1), "horas/semana")
horas.sort_values("Diferenca", ascending=False)[["Local", "Diferenca"]].head(5)